# AeroVision Lab — saved-model evaluation
This notebook does **not train**. It evaluates the saved `colab-pilot-v1` model
at prediction limits 300, 600 and 1000 using the same 200 validation images.
Select a hosted GPU runtime, mount the same Google Drive, then run all cells.
Keep private: source code is embedded. No GitHub token is needed.


In [ ]:
import os, sys, pathlib, subprocess, json, csv, zipfile, io, base64, hashlib
from google.colab import drive
import torch
assert torch.cuda.is_available(), 'Select a hosted GPU runtime, then rerun this cell.'
print('Hosted GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
PERSIST = pathlib.Path('/content/drive/MyDrive/AeroVisionLab')
PERSIST.mkdir(parents=True, exist_ok=True)
PROJECT = pathlib.Path('/content/aerovision-lab')
PROJECT.mkdir(exist_ok=True)
os.chdir(PROJECT)
os.environ['YOLO_CONFIG_DIR'] = str(PROJECT / '.state/ultralytics')


Source snapshot SHA-256: `47ca672dc5d1f9b25cd2ab98391c810d7977f8c7eb95acd8711b77bde4eb480f`. The next cell installs this snapshot into the hosted runtime.

In [ ]:
bundle = base64.b64decode('UEsDBBQAAAAIAAAAIQB4tUodYAEAAFECAAAOAAAAcHlwcm9qZWN0LnRvbWxVUctqwzAQvOsrhM6xsB0IaakMPfXSQqD0UIwIsryO1ciSKslJ/fdFdsjjOLuzO7OzdTMq3WZhChEGjjz8jspDwAzXJEAcXbRWh4pttoSjhdsIeQTTYobvGHTu7QeIgiBUO29/QEaOjBggMQV4e1JBWZNp0RB0Ap9AauW0oDlBLQTplYuX6ucgtMZvuy8Mfw68GsDEgDvrsQCvhMYhETLbJB3cQgSZRsn1hMxNsV92VWxNiyJJODAtGKkuF446eqGnqGRgbEvXtCy3ZIWJGQc3Vayg5Wb1sk6VndLanitW5DOavl8/3iu2ScBN0kp7yamkOU1RXSOgdj5J6OxenaMWTrMFN0UIsWKzrh+7rmI5fXrYsOQSOLqF+BgplVo9D0KZFH0yQu8e44Q8igME2inTcnTuwaeP1CR4OcvMA4sNqozaL4YDR6niROyXrBIKt4FklSOtDGQazCH2mOGiyNE/UEsDBBQAAAAIAAAAIQDrzMOnIQwAAG8aAAAJAAAAUkVBRE1FLm1kfVltb9w2Ev4uQP9hgH64xtjVrt3E18a9AobTpkaT2EjiFIeiOHOpkcQzxVFJatfbX3+Yod7ipvclSHa55Lw888wzk6/gEj19MsGQgzdql2d59qFV1q5p91/UEUqMqCN/i48detOiiwHIwScTXnly+BIwRLWzJjSgwGPnqey12VmEnQpojcMVxAYdGLfHEE2tIoLHQLbne1cQjcUSjKvQo9MIypVQYmfpyI8VbNLHxgQwARTfUmKHrkQXwbSdRT6kxMIQlY9YQuWphaC9irop4B1B9Mo442p+trcxQKP2CDtEB8lcLOGI6amvvoIPUdUIpy8nb5AdgKjCA584LeCTsqZkP/hzdHvjybEdYvvr2zs4mMjhCA35CKGlBwTfuyLPzgq49dgpj1ME02F87KzRJoK2KgRoVdcZV68gdNZEaJUzFYbIIShBOUeDz7pB/RCKPPumgI/sJij4982bm9PTMMUfVITz51vozCPaUMAH9l4ysMcSAsZoXB1WMAZWH2GPniERVhAQyxU0ypcHNpqflzc7Mi5Co0KD/PzzAn7cK9tzUMjBPgWILSxVVBf8HPo9QsQQ1yXuoSIPlXHKAqbf8VlVRfQQ0DLmXJ3CO+GOkxVJky3y7EUBV9RKHM+fb1fw3flWbDs9+3YLxnV9XGAsjPnQ5KIny3ibMBEw9l0BVw1RQFCwY9hAMH9ydlWEynDUrQWd3is5kykPHjV5TscRStwb8WHAK0JlfIhQk7KM3EhfQJNcMpcP9R7o4KBFFXovwA4FvGV72FDlKDbon6D+HwHwUekIrm936KVMHEVQIfQtlp9j+uwl/9rTHvnj9RTCQ0MW16blM2wT7dHbhMC/lGcf+FNNLpgQOStBk0/2lTOKQ991HoPwyoiwgp+8XrAAek8+gEPl5RnYUe9K5Q2G1WhDooOer+WftOhr42q56j12qbwWfOVRK2tX0KFfp0q6vF1BpWxA6CiYaPZ8eYst+aPcbVUUyCtLrg6mRHmZE95e3so7n9Cb6si+kfBLSyVa6DyWRqgxgWGAMS64i62oyLfKaZRE/LhHfxxTICfaPkTY4QyukTkU1+ZUweQENEG1KAWVokK650uwXFaGpDtRATmoPCK8JqotwhXZxPAMT6u6SB2Epo8BSoZd70r0wl6WVLmCsCDORgVoiU2KBAoaCvzq69u7It1GtgRLWln5ve+dY6AG8BiNx/ICrt5cp9s2g6fg6AAlVqq3kS+9kruEXV/RwbEJ8NvcnP4jxhemO7rd7187irgjegibL594lpj2pkMHTYxdeLnZaPmaeUh53RS1xKTQ1G4GUpP6Pzn5iYH4A9x1YsL40slJ4tiBJk5O3vcumpZPXjXK1cLv8kE8dvzp69u7kxO5uRdaGUI2nEqM+b5PadVobQBOmOcUCMV3gpExda+82SMorTEEOHA37Ty1XRwKXPrjaCpgu8MygGIWN5Vhjqfea4TgVBcaipJbR/DaxJ/7HUR64PbM2fqj53QVefYLYgeGKdfsVcQCXqmoAnLPVTUG7rVHxpcEfWy2ECJ5VeMFuwnKR1MpnXpWns1dIwi6B7zdvz0m7xapfKN2GybZzT2g0g1gR9zJ3yMDa+gLeTb563HXG1uGuURCv2NbE0czD4p06F1lnAkNB0TKazapyLPrCMoGkq/SVYlXh9jtelda5CSJtQX8JJV1ezdmRR5L2c0zayrkvwXYK3+8gPhZgriamKOPyLWPrK1Q9wwP4+DIbSCFVWlN/SSBEDpjKTKewtDj3Qq+3W5TYcEGzrbbZedNqVotev9q6G5necbmnm1TaEMB1zHJp5DU09XN1c268yg3MyuhqRvuRmwGKm8N+qHgRd3kGZe1xci1zrFL3yWDD4rBHdH7nhELO6zI49A9pXPA9Osh1wONTZIiZZZzNlxpYkPS40eK4l8MNbukjeEGbsz/jzueHnsmlJT4kiOVJMnnzAdvOTkz6FKN9oE9PMrHYivHbJH8oRASwhZK6sPPl+uzF+cCoxl85E0tIin15pZL24fGdBuWMujiKs9EXA+sGhZtCaxpWbt8s92u4Hw76KPtdpt6jDyS3CafzkJlHrHMMxXlXAGvRP2TB+ojSyq+oba0U1YaMY5PMMghUo0sUAQAxrNSN3tT9srmGVYVChMI8UcwgbjvlglQ32y3ozRLNMSSAsuBIVLt4GG2luurNYnGxnbCPplEp3yWNRUWcGntJPo9tlwmS2lqVO0oRKNZm+TZG05g6HedJynpwWvDdOdRtan9yRNDNgv4iUO21uQqw9IZYW9Cz6qWtU2e4aNiZA+6va49ivKpPWJ5XLdSjVLkKTaBRwM+MFUhXN5Ci9EnG5OwZEDcSz9bC8TW+9P1rKLX+7PiT9Pdj0xGY/cQyZBaaxD97XFv8DAW2y+q5lYjZbZOGmeW0AsZzsevW5Fe918spnRRKqR7rnyWDEMjGV+ZA5iKNs8Gh6reWvbnfJuoab07PV8PKUxuKW5tJyfDhScn45Vjh5JIx6h0k2eGhXCaBwr40TEo4JqpyI0HHai9Mla+GdncomeYDXOrX3bpPBvbtEyWn7XnVviA/N/3VUH71AYSEUgBL5lABPhY90/pfJi4VotMLo6Qs8dVYqyJEJbafjFky0i4+e58u5GRKVHC00lcDhV5duMsd/sUIROE5BY67X7zICnYHMg/GFdvFHraCxjWM4Q+y2KeCcLZ9VRlm8pwmXTKoU0KZ+wJTwgUSsLUPgfyH/F7dXvHYxha6kaYfpw60XJIT0QQFonnYhj6PJYy77lS7uNxNbCn/GCeMRQ4BDt0ummVfwiSbj44KmVR6iYM+jplnG/BEhrD+siwOb8aV9Ih5NnSLkEK98QkBO5L0mEzYEeGwbX4UsTHeH8xSASPUXwxMeRZ59G4EJUMuLfHj+R1sxhUU88ZR3tQOvbK2mNqWcbxHQL2FqPiIWMM7Nt5Hp0GjTQ/snO8IFg/Pxs1Vxj4jqrKaKPsIE7YiiWW5y2OPa5GL/IMZNKO6CAqX+OwDWG8T4uHRF1MtlFCmZA9tENQ2lMIaWvChAkJPBNm5ml7NkEjUDXwtj1CMK2xykPQ6GS3sYabsRx5dOAJEmZRwLFLmli2SmMfT3XKYZ3l0LjFSYNw7aTJSblz3xxH2B098gMegVoTp4UWK74ZZik8IshkNSahS+14oIIq4XCngtHz7zS5AQAv2bAjaOV4/uT+xk8p/oV+qD1P4sW8IzCBXFKe3VGTpkhkZQ84yZ8hCEIkrXp8hTH8S6TH3BvZyG2x3Z6K0+/efoBruoNt8c9iiobHOoFTBCKvENKAq8JipNeeDmWK05xiXg8uR2KZoB5NK+FYSfKFbUZcTuu3SVaIu78+WYUkTlyoKh7nZvQ5jEx6f1k5pZkH5OepbMXfaRUCH5/eq8oyjOJK3Fwr2bi9e/uhgMs47kNSDM9WoJib16Lr0/JElXteNARQO5bGL07Pxo0f+/VmWHMYp21fJqE46BzZMwqzj8y/Gvcs0wjFQ7pXLlToV7DrI5uBj8NVEq01EziUqIk1fZHWoBKaAOHodOPJmT8HwMtYxxoDDsqzruo7yeQH/KPn0DJv8HVzL6rUnnofwNJh3N7wDqpvuwT3PXroUD1AbDz1dcMdn/3m9c/l7Yvt6vM90eXt5kurIjGuxdIot/nuRWzWHXrNBlnemEgEpdJYcHouTpcAv1gGrcZhSShgNfIi/yOx4MC/Yt3HBnk74GXNaUlkx3FJlEPPvOBxfdqd8vwoc+MoApAVQ+rKTMzveUEpM/Zy6EZZPg2z9EfeR2qJmizV5+nM944JmVsVji0hz0rjZRJIOzOugftE7Ou17KO+P6rW/gDrtWMu+J7/5H+l6Ru+71RsNpE2VoVYdPGH+yLP3uEhrVI9SkoUi0wTZPHLbWh+kgXknkwJwVjpGJJuTsBiqT9tOqWPDwytqZNV5/Xf/q/BRWI1poY0Uw/rPan9kTXR1byFYx7Os9/ubPTKHnls+P3rcb3Ezbro529ks/RsGuc4SAyDUfXk2W8TAUVU7XxRbWLT7+Tn44npL+tB4z4rBjcXDcljni3wMMmrhbW8FMYYoDU8nIiAHe0iP8GW0aipTb2n4Kk6TT8QqIrTfwKMP4zo28BkayVPafTzGDpe4TMCqe3IySo7z/4HUEsDBBQAAAAIAAAAIQBMKeznMgAAADAAAAAaAAAAc3JjL2Flcm92aXNpb24vX19pbml0X18ucHlTUlLyzEtJLUjNS0nNK1FITC3KTMxRSEktSU0uyczPU0itKEgtysxNzSsp1lNSUuICAFBLAwQUAAAACAAAACEAukkc5B8DAADsCQAAFQAAAHNyYy9hZXJvdmlzaW9uL2NsaS5weZ1W30vdMBR+718R8pQrtYqTMQZ9kI2JICpjGwyRktue3mZrkyw5rbq/fiRNq15zq/OpTXq+7/xMvtZGdaQo6h57A0VBRKeVQcKlVMhRKGmTZNozG82NhWn9yyo5vSub1I5Jc2xasZ5orjg2SZJUUJOOC8lWHxNCCFE2AzkIo2RmASuoed8ioz8vzy+LT5cXX85Oi89nX2lKLBrmOLLytmIrckBoZpEjUPfat2h4e4+itHS18sQ+PkPyOdbsxGz6DiRe+S+sAlsaoV1iOT0Bo34IK5Qk53xNbMfblpxefSdaaGiFBDqy2n5N8sCd8aoqbL8eV9YRYk5L1XVcVjQlBv70wkCVfzM9zHCPGiGMVqpEZWhKGmh1Tq+MkEgGMC4QS7isfAxC1sp0vgUhDG1Ak/wZndvmZorVrfxnHhJndH/fKIU0JXivIXflTEmouV8xWnHkcwljBGi4kBODkPhA8OHwcAE38DaKOlpEWYAqCjs+GlG1wEghxiBni2e0PssnVYg0KwYErcrG7khkN0x0G/s3inp/vABbcyybuLPdIMk7iE9ftBIwiNLZT8y01P1C5V7TkGegW2V+g4mXbSF9A7b3uUy2F0rCfFY4NgQVabnFTGMIGYbINJSqe3QsYHjbOERwtyA2Ddq3QF8xEREUihb+G6QGMC3XE65uFX9c/+xod10WhiNi3opO4FKTY8H1qPvFG8n00h6MLRTW3YA7uTp+t19BPIJ30zUTwcHA20Wwm7vdXlXlSlQ2SpRg82t62yjfJeq65Q4LXSts6M2jMvqNkZGbjSU5GbixLOiKfzgfloVkg6iMamYzrTSbhWa0EPWDUU4mYRk11p8vJ8kZ3GkwwkU+yXIQX7eVzMba6RBzop5Vfacte2TFVikRsgLpLqFQlHbL+6RD2+7dOZscB5ttp2GbuTyvqVerm9RnfR1u9XnpFGVe+HvpZldAI/LlaqCzewjJL9nennMxMVt4mSZcOA9EbkaeNM8PzUgZ+uuzHl2l3t7mbHuUVq7NI1cepspHRJjbTFerJElETYrCaUBReKuicL9bRRGyH/+9kn9QSwMEFAAAAAgAAAAhAH4U6ex+CwAAFyYAABYAAABzcmMvYWVyb3Zpc2lvbi9kYXRhLnB5vVlpj+O4Ef3uX1FgEKw0o5bdvQcmzihAgpkFBtjdWcweQOIYBi1RNqdlUiEpH2v4vwdFUqeP7t4c+jAji2RVsapY9R6bEPJO7kQhaQZS8RUXtIBfuX6npGBAVbrmW6aBigw29JFBxgxTGy64NjwFvaFFAbpaamZ0PBr9JCuVsimsjSn1dDxecbOulnEqN+NaZvNy944aqpkZ/dUpgQ1XSqop/FIYRYuD4akGqq1k+EFCVWqjGN1AKsWWKc2lgFRmDLgGtmdpZVgWjwgho1Gu5AYWi7wylWKLBfBNKZUBKoQ01HAp9Gjkv62pXhd8Wf/8rKWo3xUVmdzUvypVFHwZK/avimlTf/2NlzkvmNNYUoOyanU/UrNu9BzopnCzfvzwXT3jw4au2Gj0w1+/f/8TJDAbAQCQkmVMG8WpIFH9RZYFq38teXpI258pVfXrtl1iVJU+tj/6S+hOcLG6G35eVrp+3UgjUex89P2HT58+foIEyIWgVm2oxi5UY8UKRjXT48yn1Xg7iSfxhIxGo4zlkPEV0yZAX02ti0K4+wtoo6ZW8xqSOiaxXtOHr78JQjuw42ZtPRzLkomAqCUJgWrI3Tp8cqlgWcj0EbgAbpgKCrpZZnQKeawYzYL7ycNX8ArwvzCCJSFhu9gqj6syo4YFVorTq5iplIB1vGZ7b3vot1JSpdlCyZ0ODNubKe4igh3PcGtcmAjWjK/Wxv6w2zRVWbBZwbWJAP+dO/2EkE9OzZYWPAND1YoZd+r4SkjFMlBshYmL8s0alnLPNKQFL0uWgZFg1gw4plOMJwCFeiFRIyGB2TyC2XxU+6rggqGr0PhYlwU3+EUHHa/wHIQ0dmaMOVl2B/FJpTBcVKz5qOQONeWFpCbY1otCq3CL2pws1BaQiISoopk27youmAiU3IXwFt70lSrKNYNfaVGx91gzgpy835csNSwD6/G2guWcFZmewhHVnoiLKT77CA4R7CJYR6BTqVgEKTVsJdUBEtzFbPpNa85+EsFhAgls6D6YxJMI9mHU/jh05N5HcLjHmVwEzgs2I8II9vAadrisGXH5EUZwgNewboXwHPb38DaB/QSkQnlvEzhMnnD9Uu7R9c7WCCXcOcPx5TDpOdduGZIErAIuTFDvPsQgBZMI7u8HsfaJFNOyZCILlnLfWswKnoM1sy/rbQL3A7t9YtZijqSeveAZmQ7W38F9BGS5lHsyxQ2euio1ezIvfhGPQu5EG9tj/VYngz/gw+NSlytfxQKbsf6IKylNp3jhizMEB+LNY8ZVUFLFhNHJz6piEbA912YhH+1Ppxf7hqAbBgnkpM7Xh8n9n+7evf/57mj1neLfeOmOM/YEm5jSwLhZPOocUpwRW0U6CDGq+NE3qJjrhX8NcF4nsiVVKNiuxtqy0FWe831AUHeMo51TUyoMT95gBi5WcKyNOZEI8qLS684u8bEVq98/40oVto7n5Og6zGnck2P4hsnKJN9MbJVXTJdSaBZZc30L2J23AOspb5steZPe0PUOUSt4XqOwsuKd4v1eccWK14mtZVcm8jbJWAZ/hODLB6+6tsAe03MDuvFofTeFY0fceAy9zZzge/63K2HyaXQhY9Dn4ZNH7YNw3auj3uPHKRxRRLf82jAqVhY09RnZdvnagn/w8tsmYTHQtbhGyh/gY8kQy9j+lzKBcAQyrlhq8LDTQktQ7DNLjQYuUrkpC2ZYY6KOh7v3KmL0JXboYb+71Ho2pTl0tormDooLfqpBg2JYGoK2hkRgFOVikcpKOKwACbyZTCJEA/2vD/hVM5bVH756uFCBfJmIFdOy2DIPoGRlyqpbQgjC+Dtz7Gg/3W2PjdLTnT6irpMrQDyHwMsYA9lQwXM8yIiYSdjUnU5d8an5HkcwRI4mTOHopFyrFt5lrarM8YQYIbQzpVYOCRybdQRtJVPrnqj9mhYITDWZgoXZnRFbYXHgeOp8rSnP8HuppJGpLMgUyN8/fvexAWlyw41vGg6W/RnYlhaVJRp+QEOq5K4BcchnDMg85ynv0q0NQ0iuY4/CTw1Ss6ZGYMNi23NAbNSwULbRCyMIyJYWpJM5YScifmsLRNFY8XutzXU19HfjgTYmtcNn7eB8ZpfNMQYETwuZ9hTYExQBcSCeTGvg350Tuh3a7ELsusi4iqCgS1bgKySdLLATNIFx7Yx2yC5oh85lPrcl49Nof8kixVKpMh11KWaDJBZ+tAHgDQi/WvF6TrpY+Zr9oVgtlWFZIGyuCEyQ8zKG55eM3ZIxwTnC8gsRF3LHVBDGTGQarQlI/LlckbDfqnx9nMBbn4ZvXVdzEi/0xwuV8pNr/iyDo6sw4GGO30oEdEt5QZcFg2NHeLdx4JOupWai3bhj6vEn+1+ABSCMNcVa7yX4ozPYEnrL5QjPItiwzZIhFAYmqg1TyAOdogiGSHiA37D8Bm59aF1+jgZctWoycgjhhnMdtlgsD4bpOhscNPFqznGETSR7o+DgUdM1rdLL8MEyk5qm1vbFmv92blWTcPGWKZ4ffFfpPlSIhfdi4t3ZtPg29SIg484xGZMwVp4OxgRdPZvM4TWQ2OyNK/fdR9HdAvkqJNBzS6s7jDOG90IBqUx+9+ZO89Ugfa6w4w6dr7VEfQ+di3Glx57s8wSRyquxJNspvByHloxC4mfOHOk5F9uqrRnUxSnWAnL0wr7ocKwv5ic4BpaMwhgeQhj7O4v4TY4jlovWI/4Gww3tBlPX/Qnn0cLn3Gn4dDLgyW20Pf7SQyxrxGrRkRnCa7iPnljnjz5yzroK3Fzx6pXz5hNyqWKUTGEHr2D9lAna4gIyhcn1mW2TvO3WoG2cY1eS6hITDogdnqww9DUG8/yy78k/BYk/Sy6cZB1GwEQqkfQl7nCdR/zcLN/9bkb5eoTJM+NDcK8Lj0Pqfd+YbtMYo2TP9/V5Lr3J1Of5jZlDoGPr7+X55xG9QAv74KG9LbmUtkAs8sRPbtnpEnbzgLeD3LzwZm4qU9kD1XZnXORyAIfddwfJpg386Q93jiNiww4y6s/zpYlbUbOjj3gENaoUJ9epIwdt2u5sAX04b8W1fm2ZSk7qqxTcXE1XOqmPX+Ks2pQ6wBkXkjy8Kba+FbspeRDNm0pq4vSjY4pZDZLwEhOvRL2IFjQdh8XvVPcdlxXnPMuRBrxnt/jxqGZ1+s6try0MupA4nnXMnaN5DlQcgu5ibHRd2TeFIVWZd5DVGWIkHzImDE9p4fZqrzyZMA6Ga6Cpklo7jRbM2gsI6wTvz1SKnK/6PBFBNZl6/hBTvSil5vugc1b9Nqd1io892WonoOntsOVc7SDmLeZyxlMTDNPVq3Ee9EzsFte1XKqTVDgSa5qzBWZW4DYYWSS8eGQHnXxLC82uJtgNCn85c+tZEXCB0Ugersr2zB0t7l92LNq49O49encbX3/15om7DULqM9EJNNBCCjYFupWdu6f6SsgGDn94sOgqiciZYiJt/1bykluTVvXd9njxoqQB+ef81aZKf9ZzeeY19u4Z/xX2/juZ6e9kpf83RvoiNnpeV1oi2kkkf7uiMWiaZ81fv2EpK5F1r0OeJo4vJqbnHPMCZbzNM38Hb3wpZ3wBX7zBFakQ/xuCeJvUDfniRaJ4k/E9k9E9l9vY/nAVUPIseRmpqTMyeRotP81kkMQkfSY6e5jDqwE7nX05v2GQYzfJFW5zjnn7X54iDtZ93e1CwwESexbcDaSNZ9KLauLBPDjcknQRe9+GwR3YZUzuDDnzvkNfiV/kBXuE3Q89AuykC6/dUUjOcHUnGZKLgLrF0sms8U8E1ieWCTZQGm+urqPpM7CwpcV/CqAH0v6ruPl5sKZxU0eD9RJW5sT+4cBdIevk6CGeV3zCCn2R/A5w5i0o5yZdNGiA6qxR2GaTc5SKLSvpIU8bTJ1chps14MRnuIFurD14czaO/g1QSwMEFAAAAAgAAAAhABcHFKG2BgAAwxEAAB0AAABzcmMvYWVyb3Zpc2lvbi9kaWFnbm9zdGljcy5weZ1XS4/jNhK++1cUmMNIM7La6ezswbMaYDeLBAMEmcYmOQmGQUslm2mKFEj6hYb/e1AkZUl29+4mPrRaxWI9P35FMca+dmi4E2o777RQDtAYbcDgQeDxE9TCOqEqB43RLXz/9fuv8M8nwAOXe+6EVjljbDbzi+t1s3d7g+s1iLbTxgFXSjuvZmezKPvdahX0Ky0lVn6131Bjw/fS1aJyQafjbifFpl9/4m43m81qbEAf0EjeJTyDTQY1umBqzQ3y4gcuLabLGQDAKYNzBscMdlAADyKSkZCkJN54sVAOjQ12oICWn5JFBq1QyQk+kIWTfx5TmPvFE0nSFN6PVc/wAXZknp67XvVMkjT1bmpUuhWKO22ggCO8hx2I5iYFQGkxLpJP+mcH80mM3ppBtzdqGvvDxIe3Pbx+hkUwvsgXsZYtd9Vu7RtvE8fNFp3NoDNIjaD+ZCC2ShusM6i0akSNqsJikT9+zEDofbHIP8ZqM8Z+qbTBuTY1hg2SWzvnR24QtgaxPgd/Qm2h0QYOwu65hFrwrdLWicqCVvLsgUUW98qrYw0FWHSJ4WqLiUTVR5rGunpHohFes1x5GTkY8gChwGrjsE4myT3juZC83dQcuiV0JbOUAltldAyorMWvZt/jySOlGVm9qsM/RtUZlH1sWjmh9ngVVrrtuBMbiRSs8IEKim9IVzQQMyzFqmQVd7jV5rwWNVtBUUwimCyG1Om3QesikgeHk3zF8nqSxvY2G32i/McBBFma9ae0+FkrTMc18e6EBaUd0CJwVf9f5mnj4AE+F4SqaQWvdckNtvqACe0ZvNPvWShqPXNdQA79UIoGuDon/y2MjT7dUYjvOEWyyD/67mz0ifoTD8IIDGPXcXXs3+Lrqs0oygG6Oe86VHVCMY5izfy2gv70aNd75SwU8ELCJdh9m3Qlo5cAD+/H45/CHjykXuhXhYKSqpX5aLIh/NVl5KNkjSKTQGfu2oZ0zD5BMRt5yaAcsDOgO56+wcoqUlCYN4ndbyy6pSd64pq240ZYrQbJDfvgibedRFv8PXbEj42nLz/1I+NLy7eYhce/DT/OvJYze0fET9Mol5rXNom+4QHYgcu80pXOaZmluUFerx2eXBKLH+tE1TclE2Tbn7wlmJJt9AktC0kbSvptJ9HOa35CB2INoRiPxkSKHvrkg5MPn1DJRhOXrQbc9a3g42BXPdZ4sKV4ix5QVclCMlXJSBhzqUZ+It8ISjREapDK7fdf3bLOaKcrLdkS2I+B+y1vce5xAl/0b59pdAB30IgT1qP2fvIk0t83+oHBssH2oMuWo40jjVbXaNkSXi5BGALVe9ftiRUHdFE7/PSbBxiykWLePtfCJHgS1q31c+CFa+3JRThGx52WSGfICUknaCj+aNS82cfrrLrFyyTIhr2Qw0s+MvkadqaEM1Iuu1cB0A1BGKy0qe0wQfvI/LYRAvz7BGQDY0yJoBXW+rMyuWZMto0x6g0HBK4ml5Cblf7sbNElo5UMSppRAyAmjqYDIyY74VxvKoP37+9TKe6zKsIj0oJPQzsuAy/jOdCyKZ/xPOKD6DYSMZ7vebhRr7DxYL1kncFKWKEDMfdS17EVPPhx/202lX64vjYdW92GWzKDFZfyL1hTY2uRBMp49FYlPVcTTvAnM+xmy2gmm65GaC2hfHlewiGUKYODr10uHLY2Sem+8exJwldvaA1VLnSFrS53RV8NvoaqfgO/8BZ7iNcWeGW0tf5020/QCGPdddIAXWSrnbaoYHOOxHXgUtSed8FffIXa5pPjM46hXPa2bk5PCKAIwyrXHarRvAhVeThwyeCB5kwjJK4DP6d5pdUBjUvYf378F5uivDb82BulCZjTn4DzqaI/5g5bH+qkpDdx3n1W0a7+OnWnWWnpv3ReCDtLYFK0nigb/0YAH2F9CWxr+JldymAz3GfujVJSucHKcbWVmNC32DmD+J3mv73SjBhcCoWFjyCDo6jdrnhMXzfmufNuiX5knE7B44JMz+HbxzTNiIz9xAyBvhvdwN+tVhd4CWL/afButcwfmwvLoBFShnDuPN33IjBi7EYP6f/RibDnzV78qbKxM0qpj+yNyt3aWmSwyAKGc78hg8dFGnNmG8mr51eQ+Ubdk79l8F06JQZflzgCl/Drk/+cpLn4wxPQbeyhHzUhbni43tMIUp/8TChehslwGV0nrvZ9sMedcHizOg09pGn5AZN4ofDj2ZTvqP/LxXeXeZzVv3fbmPagGW4b19l9NMJhGN5+9tf7trNJJNMMhKpRueIxTpl4547Lsz8AUEsDBBQAAAAIAAAAIQC5/4yQ4gsAAHUjAAAcAAAAc3JjL2Flcm92aXNpb24vZXhwZXJpbWVudC5wecUaa2/jNvJ7fsUcFwvIW0dxsk3v6lYFFt3ttUDbXfSJnhEIjDS22UikSlKOvbn898OQlET5ke1dP5ywa0nkzHA4b47CGPux5lUF/3z3M1jNhRRyBVyWUKi64ZrfVghfvv3y7bmxuwoBN7xquRVKpoyxs7OlVjXk+bK1rcY8B1E3SlvgUirrwMzZWRjzt0rcpjVaXnLLu5nfjZLdc1Nxu1S67t5Ne9toVaAx3YgVNfp1G27XlbjtFn3H7bpfbcfr6uzsrMQloNwIrWSN0iYTOP8CSlHY+RkAdJhW6WJ95kas3vkpugpV18JCFnGRFmss7nLV2qa1SQ9J14KthGVTYBo35w3XBunl6zevXrObKVjc2uwn3eIUjC1R6ywi+vrNL9///O23PblJaqwWTTJxI7gtsLGQvP3xjdZKT2N2vuRVheU7/+amJ0f4Z628k+peMjel0bZawkMPx5qdXSvJ5r34Uz+Sb1AboWQymUbAAWYEHh5GgH55Ng98xCR4ccdXaNg84oKuZn7ETtKOi8bLo7uWSkMDQkLC2spqXu2sKAwJ3Sm0f9gIwqZX2dbNjh6aXaEKZZWqDBuIPsbMtyXP+YaLinyAzb2RpDScCjPMjHa8atox5AptXuJGFJhLXmMym4BYniYFWBmE75XEPU56qkEUDtnDPAYzd97rLZKENnf+4EGwUcXazEHIoARRr8z76P2W22IdvROvczBW+1e/g2jAIJYOHDL4+MqP3St9h9qvAhnM/KhG0wZa8G+3M8jCBp0vEpNHfNGpl3w8UmwH8dvbb996Z/VeCJmjkjDdSsMmcOHY9zSXASbFrTDWJBMX26SyHWO9mDUXBuErUeEbB+p8KVmyH1oJvNLIyx14InN48EQfP4NirZRB4FLZNWq3MCgN5+eefLCtQsmlWEHmIs8QNEhNmbE6cUau0ahqg8kkMiivuMzfhmGnvsz9DoNOh5n7HQa95jJ/G4ZJfxn9DENBfVm4DxO1KrHK2E5V6vJSpo1lw5xqrKjFe9QZe1Xy+tdoqtKzbJbOZpfREihWa/KHgu/c3Ox6mJS3Zp95Xjc+Yg4BjRdrzL7ilYkGS7SoayGFsaLYQ2i4FSgLPBBhUSmDea0MF0U2i3frR9KINVPwCrNZ+nIYWlaiqfQYrNHqdyxs5u0w2hmvMaOfSK9kSLm6299KUylr9rZg+AbzBrVQZRYJs+bbvESbvZwF7r2pPRsECaUwFFac41SiEBZWmpcCpQVeFG3dVi5Fz8GuETT+0aKxWHpDAmEAtx4tdZQpGkMWZ40osbL5OM1G4asn7BIBOUI0aSzXFsu8tQWFOFEjZb4lPSTs+W/nz+vz5+VPz7+eP/9u/vzHf7Gph1nVDqJzlUf3GweBtL4rhU56OZNIvYAalCXVOAdBY8keSEmPQ9qhuoTFSOm9FhZzSuUJTaZlWzcmIYQpCFmitNnVZAooC0XwGWvt8vwfIQo8FfiHOBSBaDRo8wb5XV5jrfQuN5ZTEPMxmEQHmRdIg3qZF6qVFnWYd34LmQuXiQ9HFJpiR+75OgiGaNqKiDsiqU8tHijUMOPAsh89Qt1SmQ/RfPHi4W4OG5fK76awoWzujSQVFmsK2GIJd/C3DJjDY4+TI6Hf006dq5RCBwGg5QtmsFCyNOzmuKjg3Avyv9CQp+vUQhrSGyzz253FsMZAgFw0KK4DDLpJAvsXwHBLvk1e4+1t8teMrNFCjvAeyMKZS8Bh1ckUeqnM96X0OJnCsmrNOnKZUC967FBt+OMBHq83fKA38dBQcUAGn3wcYpYVFR4Mqg3qijdzWFaKu1IivTqoQ6ionYUgW4la2L26wzPrORj5+oXnXDhZe9gQSjsKfUSlw04+mtyrYDo3M3OwbVPhggokSNOUDCFh92tVuTMAbbOk1Y4fPwrV7Ebvm6v41RWtwA3I5olCKapoU3rqwOjwdhqKthhD0vuHi68eICXNd1OlWKGx0SRtu0dsNNLGzw4kC9n4VelOIZ1PUrnmxExzXO4SenGjrvbfF7MLJg5ESI832S/zfuFVi77CY985ynVrLNwiOGJUJF4o7cyzHKJkx+HncDkjVkZsf96b0RNr/YB/tEJjT+mLjEhRSTqi9UXWQbCJF5hpsIDMnWpTw5eYV4qXXd3ISx8tDiJCcF+l+kBJdBaMDs7sxs9a3ToPcSGDqJokcRgXwDa8clbSRaYPLiVqOtVRFCSqC+bf2c1i7nz0hqTonvxBZw/KkygJ/0EvmCjZjdOlJkV6IJ/p9/AgC7PxZNSCcBAL7mhxonUURCyBB5q5W5oWLY3n6pnzDii0ui9hzWVZUQ0hDHDZl0nAm0arragdSbAKfhHmtaaAIVZSaSxB44pW8wVVN3ha9gHiT4tf4tbmgijWfJskPJLhyX1PKJcveVvZbDaBj+DSOzCJXd07GXgmBrOmekHdj0QVfFGUZgCjq1DSCtn6w1hH91ZtCdjRuFVb0uAYi4AKbnGl9C5iOwyJQ4RTak95Q5XbuE3TXeOz2P4lyixIc3oaJggg2xPHaYxuV4TUPXslnca5vVXb7FZtT0NwjZwgFlc38ILEu3j5BD1hnBXHZ4n4GrdYjo90dvZRFsxlRQGGPCTUNyvrMoNx0Z3SW1oiNvSQOD31UIVGbvEbWeL2WNlKRUsoJoKJh+O8L++pApHdgWk4VMWVi8s9/8OpJayamzW/uv6EzUN267mJQGsuxRKNPYDtXLkDCH4c98G0sqpQFZsDi7qsr95NQ6gZhxRyjL1Q8pnzPbVcikLwaog4plBayFV0DGUGrRVyRUXfoe0fbSnQRTkwo5/xcCjSsnAfT1IFv3/4d4uoNpulfx8PnmhRxGfccB9Pxhkzi18GsFjUnXvS9o2iQ2ciyrEeqRCgjmRoA/pcc1BMRPld3Rt3ICWpTsGdKSh5LW6m4X8P+gx+5bqGtgHkxdoTVJKyBy9s62owvsIpmbcRJQaiPk84LoQ2zpc2V6moKRE43+gszKe/iw2vGFyEZLiY3SzYUlS+88huggM5PSwDQWFcJTuOpkMn7Htlv1KtLH31coJs1P9wFZ6rzqqpX2Hqtjr15kWyqmiTwWbAmwiQSQQzmO7rfCBfUDAv877EmI2SisZC6XIoFMZbcmN/Vnye1CnZBfl5ikfl95QMj9HeJ326O0yFopeTOxAX1G0+TIQDutnJYq2VFO8xRNfu+lDzYE+nFGmmICHrlXywbNB6sOO/qPUR9f+LgIJXdxVE8kTrYAIv4HI2m40J+GjQ4cu9yZEtf5RBhTKJZD2JjgBj31T3JsWtJZpOD0MNEizLFRPw4kUzGb6PxJT3hUkre14n8ByuriHLYHYoNN9XWLIH0uzjHB4ivMcL9+Z3M3kMHjilXWYPYReP7LCvMO6EdLTTiNunWyIkjJONkLA9ghnvhtrFnjpkVIJQ3f0DBmo95Lh3dYAX1TqH81Ht89AdUua9VOI6dn6stp3CqIqdw+Lm8dRKh/WTY95/pVXUJelO9cnKTiPMKTAqLNkRpJS++9YmFfXqG3cYi9Llaeiab1+jT3+XU7icTUcJ+uYIYniKPW+Y7BvU8fQz+GmN5HSy5LoE09Y11ztYc10W/hTvmcguZzNn/a/efdbVgKrV/UktIthhvJzNwFSioIaFNBZ5CWrprd59EYdamAq5ax2fX8IaOR0BccjQjcbCfWcM/YxhjwvWz7Ebf5z3WlxCjVzmG16JMuFRk4IuNwoZ8AWnYDAbJBi15FyHLHGgKdFKJq5f6geMeI/DB8VhYS+QRah3bhZ0pxPyYVznTRZx2O9iMZ+C/zebwtVNVEN5pOvZcbTZ02i5ob9E+MCKl0dQNRa8qg7R99Xg4agbcZIUb+hTS15U3Jhs/GmarmLBfNKmvulJNu+6HXZt7YKCMMq2Rk0GfcTnx6Ek+g5NF3mBLHZ5cz3La5N5pcsmrbEUXCYhVcXfDUdYn16PsBrUBUrKyR3mFD693sd223MNvHg93sf8PfAhsGSUCnxkPizB/GxIFMfmc27zIf7nlEFGuTKq7Y+kEPqLg9Db/UDmcC7wJ9rpdB201I860IjYiQ66Rzz7D1BLAwQUAAAACAAAACEARu5Ea5wDAADCCAAAFwAAAHNyYy9hZXJvdmlzaW9uL3RpbGVzLnB5jVVNz6M2EL7nV4w4mX0dRHZvaNnb7rHqoeoFocjgCVgFm9omgbfqf69s8xGyb9VGSsAzj2ee+UwURd+noRO1sGBFh9Cg6tHqGZjkUHfMmDN7MI3ARwdjFsGMw6DRGKFkEkXR6XTTqofr9TbaUeP1CqIflLbApFSWWaGkOZ1OHG9gLNPWkA5lY9sMhLQUjHjH5VXdUXdsyODWKWZjOH+DThhbCGnL7AQAIG4QLsPXHFJQ2l/fDlJZSN1psQRf4RIuuo9mwiD8zroRv2utNIl+VUZYcUfgokfpAjI+7lcboPHPUWjkUeytdcxYyKFnE0npyujsuQSAsTgsgAsFrUbJiWf6CcgFzqvtOA5wjXbUEozSFjkxaEnxSTPZoDfvnL3BhXqrcRCUcbzk9CEkVw9DHoJvOW1RNK39/wm249DhkuKFS7GljUwUZgq9kGSCN2+OgvcWB+m8SYPfJSj3uSkNMwi5Vj4AAqeNzhE+PcG9lw/R5RL9oJGL2hIv7BXHjoZG6VmDdJNmYKxeNY15D/nwZ9f1T8djioKsVvJ2EAg1Hs4c76J+dtKz6coxVABy+JKm9BQvLRyGwypdt17gp8cf78LPlBrMiqqYrVvkV9mbkwe3FB6Qh/AS07IBi+xzGVhqdzGHwjVNSuFBoY1LNzIuA5DnED1a1WEE2Bnc+4Y6oy4LLymu1ISGgqmVdk+/C9A7KOnyPW1FSynMKYXpQmG+uAp6Nk+zh2bs/My4GiWHsq0fH1Qxp9l8oTCl2XQp6QugMe+5/z0qXIFy9/OCV2Mu1HgUhlrl4XFULVXLl+dReUddKYP5D9aZp3txkYY0+JRBvkSa+OxtCnGDCoSBX5REt6g6lKSKXU3S7DUSK+SIm3CapxlyqBL3knC0rG5JnNSdkkj2yXHaIqNQpBQ+lyW85TClP2svFL4E7bxrPdWEDQNKThx0NxuKv+qqxOV4I7HDlt54wnVm59op2axocfNL2rt87o6wc0q/TInvnf9oQj8wSc0s8ZCYPkkC+iBaLm4ktpFwjc+jncof6Df30+CRjzlQ115xkS3NEprATdZPttYk6sYteMLR1Ci5kE3+mx7x1YRWjzBk23BValq8U3B/wI3Sfqe+i2EfIE+ycB7LpB5GEidWud1O4pX5vyiXcD7UeuvL3vJ9dBzz3Hndi6geWwf8Fa1Er4JH2UabQlRVaooyKHZbcA6G3cuclhQiTzjKAvG/D/+RzsuhS/4BUEsBAhQAFAAAAAgAAAAhAHi1Sh1gAQAAUQIAAA4AAAAAAAAAAAAAAIABAAAAAHB5cHJvamVjdC50b21sUEsBAhQAFAAAAAgAAAAhAOvMw6chDAAAbxoAAAkAAAAAAAAAAAAAAIABjAEAAFJFQURNRS5tZFBLAQIUABQAAAAIAAAAIQBMKeznMgAAADAAAAAaAAAAAAAAAAAAAACAAdQNAABzcmMvYWVyb3Zpc2lvbi9fX2luaXRfXy5weVBLAQIUABQAAAAIAAAAIQC6SRzkHwMAAOwJAAAVAAAAAAAAAAAAAACAAT4OAABzcmMvYWVyb3Zpc2lvbi9jbGkucHlQSwECFAAUAAAACAAAACEAfhTp7H4LAAAXJgAAFgAAAAAAAAAAAAAAgAGQEQAAc3JjL2Flcm92aXNpb24vZGF0YS5weVBLAQIUABQAAAAIAAAAIQAXBxShtgYAAMMRAAAdAAAAAAAAAAAAAACAAUIdAABzcmMvYWVyb3Zpc2lvbi9kaWFnbm9zdGljcy5weVBLAQIUABQAAAAIAAAAIQC5/4yQ4gsAAHUjAAAcAAAAAAAAAAAAAACAATMkAABzcmMvYWVyb3Zpc2lvbi9leHBlcmltZW50LnB5UEsBAhQAFAAAAAgAAAAhAEbuRGucAwAAwggAABcAAAAAAAAAAAAAAIABTzAAAHNyYy9hZXJvdmlzaW9uL3RpbGVzLnB5UEsFBgAAAAAIAAgAHAIAACA0AAAAAA==')
assert hashlib.sha256(bundle).hexdigest() == '47ca672dc5d1f9b25cd2ab98391c810d7977f8c7eb95acd8711b77bde4eb480f'
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.namelist():
        assert (PROJECT / member).resolve().is_relative_to(PROJECT)
    archive.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
# Editable-install path hooks are not reloaded by an already-running kernel.
sys.path.insert(0, str(PROJECT / 'src'))
# Save the source that produced the run alongside its outputs.
(PERSIST / 'source-47ca672dc5d1.zip').write_bytes(bundle)
def run(*args):
    command = [sys.executable, '-u', '-m', 'aerovision.cli', *map(str, args)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
run('doctor')


In [ ]:
# Verify the model before preparing data or doing inference.
from aerovision.data import digest
BEST = PERSIST / 'runs/colab-pilot-v1/weights/best.pt'
assert BEST.exists(), f'Checkpoint not found: {BEST}'
EXPECTED_MODEL = '3ecd854c6b6607edabc455128057b3a77a2816898a84ea405c178fd08e307194'
assert digest(BEST) == EXPECTED_MODEL, 'This is not the checkpoint from the pilot reviewed in chat.'
saved_manifest = PERSIST / 'manifest.json'
assert saved_manifest.exists(), 'Original subset manifest is missing from Drive.'
original = json.loads(saved_manifest.read_text())
run('prepare', '--train', 800, '--val', 200)
DATA = PROJECT / 'data/mini-t800-v200-s42/dataset.yaml'
subset = DATA.parent
current = json.loads((subset / 'manifest.json').read_text())
assert original['splits'] == current['splits'], 'Image membership or content differs from the pilot.'
(PERSIST / 'runs').mkdir(exist_ok=True)
if not pathlib.Path('runs').exists():
    pathlib.Path('runs').symlink_to(PERSIST / 'runs', target_is_directory=True)
assert pathlib.Path('runs').resolve() == (PERSIST / 'runs').resolve()


## Controlled prediction-limit sweep
The COCO evaluator limit stays at 1000 per category for every run. The prediction
limit varies jointly at the detector output (per tile for tiled mode) and after
global merging. This does not isolate those two stages from each other.
The 300 control uses the original prediction limits, but a different evaluator
limit from the original report. Compare within this new sweep.


In [ ]:
import pandas as pd
rows = []
DESTINATIONS = []
for cap in [300, 600, 1000]:
    destination = pathlib.Path('runs') / f'colab-pilot-v1-cap-{cap}'
    print(f'\nEvaluating prediction cap {cap}', flush=True)
    run('compare', '--data', DATA, '--weights', BEST, '--max-det', cap,
        '--eval-max-det', 1000, '--output', destination, '--device', '0')
    report = json.loads((destination / 'comparison.json').read_text())
    for mode, values in report['modes'].items():
        rows.append(dict(cap=cap, mode=mode, AP=values['ap'], AP50=values['ap50'],
            AP_small=values['ap_small'], AR_small=values['recall_small'],
            p50_ms=values['latency_p50_ms'], p95_ms=values['latency_p95_ms'],
            capped_images=values['images_at_prediction_cap']))
    DESTINATIONS.append(destination)
table = pd.DataFrame(rows)
display(table)
SWEEP = PERSIST / 'runs/colab-pilot-v1-cap-sweep'
SWEEP.mkdir(exist_ok=True)
table.to_csv(SWEEP / 'summary.csv', index=False)


In [ ]:
# Fixed confidence diagnostic, separate from COCO AP.
from aerovision.diagnostics import review
from IPython.display import display, Image as DisplayImage
for destination in DESTINATIONS:
    errors = review(subset, destination, confidence=0.25, examples=6)
    print(destination.name, {m: v['totals'] for m, v in errors['modes'].items()})
# Same first six validation images across modes; these are not cherry-picked successes.
for path in sorted((DESTINATIONS[-1] / 'error-review').glob('*.jpg'))[:4]:
    display(DisplayImage(filename=str(path), width=1000))


In [ ]:
# Save metrics and visual examples, excluding bulky prediction JSON and model duplicates.
export = PERSIST / 'colab-pilot-v1-evaluation-v2.zip'
with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as archive:
    for directory in [SWEEP, *DESTINATIONS]:
        for path in directory.rglob('*'):
            if path.is_file() and not path.name.endswith('.predictions.json'):
                archive.write(path, path.resolve().relative_to((PERSIST / 'runs').resolve()))
print('Saved to Drive:', export)
from google.colab import files
files.download(str(export))


Send back `colab-pilot-v1-evaluation-v2.zip`. This is a validation diagnostic,
not a final held-out test or evidence that a higher cap is better. No local GPU is used.
